# Example 11 - SA-CVA parity (FD vs CG/AAD) for EQ and COM

This notebook walks through two tiny SA-CVA portfolios (SP5 equity call, WTI quanto forward) to show how to align a finite-difference bump stack with the computation-graph AAD stack (with JIT and SIMD where available). Everything is self-contained under this folder.

- Goal: run both stacks on identical data and grids, then compare base CVA/EPE and sensitivities.
- Scope: EQ and COM spot/vol risk with IR, FX, and default curves for CPTY_A on an uncollateralised netting set.

## What lives here
- Master drivers: `Input/vre_sacva_cg_ad_eqcom.xml` (CG/AAD) and `Input/vre_sacva_fd_aad_eqcom.xml` (FD bump + AAD Greeks), both pointing to local `Input/sacva_*_eqcom/` trees.
- Portfolios: `portfolio_sacva_eqcom.xml` has one SP5 call and one WTI quanto forward, both on `CPTY_A`/`CPTY_A` netting.
- Pricing: `pricingengine_amccg.xml` carries the CG flags (domestic measure, midpoint, compact solve, lambda sweep list, FX-EQ drift gate); FD uses the classic engine set.
- Simulation/sensi: `simulation.xml`, `sensimarket.xml`, `xvasensiconfig.xml`, and `sensitivity.xml` are mirrored between stacks so grids/tenors match.
- Static data: `todaysmarket.xml`, `curveconfig.xml`, `market_20160205.txt`, `fixings_20160205.txt`, `netting.xml`, `counterparty.xml`, `collateralbalances.xml`.

## Performance and model levers (all config-driven)
- CG AAD: computation graph with pathwise adjoints; JIT and SIMD kick in automatically when available.
- Measure/scheme: domestic measure and midpoint evolution for EQ/COM/FX to stabilise quanto drift; compact solve vs lambda sweep for EQ loadings; optional FX-EQ drift gate for experiments.
- GPU: set `EXTERNAL_COMPUTE_DEVICE` to run on Metal/OpenCL/CUDA; scripts and utils handle the XML patching automatically.
- Parity hygiene: identical samples, time steps, tenors, and shift sizes across CG and FD to keep differences diagnostic.

## Config wiring at a glance
```
vre_sacva_*_eqcom.xml (CG or FD)
  |- pricingengine*.xml   -> CG flags (domestic measure, midpoint, compact/lambda, drift gate)
  |- todaysmarket.xml     -> market_20160205.txt, fixings_20160205.txt, dividends
  |- curveconfig.xml      -> discount/forecast/default/equity/commodity curve IDs
  |- simulation.xml       -> samples, time steps per year, valuation grid
  |- sensimarket.xml      -> simulated IR 1Y/5Y, credit 6M/1Y/5Y, FX/EQ/COM vols
  |- xvasensiconfig.xml   -> shift sizes/tenors for AAD grid (credit 6M/1Y/5Y, IR 1Y/5Y)
  |- sensitivity.xml      -> FD bump sizes/tenors (mirrors xvasensiconfig)
  |- portfolio/netting/counterparty/collateral
```

## Worked mapping: equity and commodity chains (alignment matters)

A tidy wiring avoids silent skews. Keep market/fixing files identical, mirror grids, and point both stacks at the same IDs before you compare CVA or deltas.

**Alignment checklist**
- Same `market_20160205.txt` + `fixings_20160205.txt` for CG and FD
- Matching curve/vol grids: IR 1Y/5Y, credit 6M/1Y/5Y, EQ/COM vol expiries
- Pricing flags aligned: domestic measure, midpoint, compact/lambda list, drift gate

**Equity SP5 call (USD pay, EUR base)**

[todaysmarket]  SP5 -> Equity/USD/SP5, vol -> EquityVolatility/USD/SP5, FX -> FX/EUR/USD

[curveconfig]   EQ-SP5 discounted on USD-FED; EUR reporting via EURUSD

[sensimarket]   dividends 6M/1Y/2Y; EQ vols 6M/5Y/10Y; FX vols 1Y/5Y

[sensi/xvasensi] EQ spot 1% rel; EQ vol 1% rel; dividend 1e-6 abs; IR 1Y/5Y; credit 6M/1Y/5Y

[pricingengine] domestic measure + midpoint; compact=true (else lambda sweep list); drift gate=false


XML anchors:
```xml
<EquityCurves id="default">
  <EquityCurve name="SP5">Equity/USD/SP5</EquityCurve>
</EquityCurves>
<EquityVolatilities id="default">
  <EquityVolatility name="SP5">EquityVolatility/USD/SP5</EquityVolatility>
</EquityVolatilities>
<EquityCurve id="EQ-SP5">
  <DiscountCurve>USD-FED</DiscountCurve>
  <ForwardQuote>Equity/USD/SP5</ForwardQuote>
</EquityCurve>
<EquityVolatility>
  <Name>SP5</Name>
  <Expiries>6M,5Y,10Y</Expiries>
</EquityVolatility>
```

**Commodity WTI quanto forward (USD pay, EUR base)**

[todaysmarket]  COMDTY_WTI_USD -> Commodity/USD/WTI_USD; vol -> CommodityVolatility/USD/WTI_USD_VOLS; FX EURUSD

[curveconfig]   WTI curve pillars ~1Y/5Y/10Y; USD discount; EUR reporting via FX

[sensimarket]   COM curve + vol at 1Y/5Y/10Y (moneyness 0,1); FX vols 1Y/5Y; credit 6M/1Y/5Y

[sensi/xvasensi] COM spot/vol 1% rel; FX spot/vol 1% rel; IR/credit 1 bp abs

[pricingengine] domestic measure + midpoint + compact solve (lambda sweep list available)

XML anchors:
```xml
<CommodityCurves id="default">
  <CommodityCurve name="COMDTY_WTI_USD">Commodity/USD/WTI_USD</CommodityCurve>
</CommodityCurves>
<CommodityVolatilities id="default">
  <CommodityVolatility name="COMDTY_WTI_USD">CommodityVolatility/USD/WTI_USD_VOLS</CommodityVolatility>
</CommodityVolatilities>
<Commodities>
  <Simulate>true</Simulate>
  <Names><Name>COMDTY_WTI_USD</Name></Names>
  <Tenors>1Y,5Y,10Y</Tenors>
</Commodities>
<CommodityVolatilities>
  <Names>
    <Name id="COMDTY_WTI_USD">
      <Expiries>1Y,5Y,10Y</Expiries>
      <Moneyness>0.0,1.0</Moneyness>
    </Name>
  </Names>
</CommodityVolatilities>
```

## Helper paths

In [1]:
from pathlib import Path
import utils
from utils import describe_master_portfolio

try:
    EXAMPLE_DIR = Path(__file__).resolve().parent  # when run via python
except (NameError, PermissionError):
    EXAMPLE_DIR = Path(utils.__file__).resolve().parent  # when run in notebook
FD_AAD_MASTER = EXAMPLE_DIR / "Input" / "vre_sacva_fd_aad_eqcom.xml"
FD_BUMP_MASTER = EXAMPLE_DIR / "Input" / "vre_sacva_fd_eqcom.xml"
FD_INPUT = EXAMPLE_DIR / "Input" / "sacva_fd_eqcom"
FD_AAD_OUTPUT = EXAMPLE_DIR / "Output" / "sacva_fd_aad_eqcom"
FD_BUMP_OUTPUT = EXAMPLE_DIR / "Output" / "sacva_fd_eqcom"

print("FD AAD master:", FD_AAD_MASTER)
print("FD bump master:", FD_BUMP_MASTER)
print("FD input:", FD_INPUT)
print("FD AAD output:", FD_AAD_OUTPUT)
print("FD bump output:", FD_BUMP_OUTPUT)


FD AAD master: /Users/fordesmith/Documents/examples/Notebooks/Example_11/Input/vre_sacva_fd_aad_eqcom.xml
FD bump master: /Users/fordesmith/Documents/examples/Notebooks/Example_11/Input/vre_sacva_fd_eqcom.xml
FD input: /Users/fordesmith/Documents/examples/Notebooks/Example_11/Input/sacva_fd_eqcom
FD AAD output: /Users/fordesmith/Documents/examples/Notebooks/Example_11/Output/sacva_fd_aad_eqcom
FD bump output: /Users/fordesmith/Documents/examples/Notebooks/Example_11/Output/sacva_fd_eqcom


## Portfolio snapshot

In [2]:
from pathlib import Path
import utils
from utils import describe_master_portfolio

try:
    EXAMPLE_DIR = Path(__file__).resolve().parent  # when run via python
except (NameError, PermissionError):
    EXAMPLE_DIR = Path(utils.__file__).resolve().parent  # when run in notebook
CG_MASTER = EXAMPLE_DIR / "Input" / "vre_sacva_cg_ad_eqcom.xml"
FD_AAD_MASTER = EXAMPLE_DIR / "Input" / "vre_sacva_fd_aad_eqcom.xml"
FD_BUMP_MASTER = EXAMPLE_DIR / "Input" / "vre_sacva_fd_eqcom.xml"
CG_INPUT = EXAMPLE_DIR / "Input" / "sacva_cg_eqcom"
FD_INPUT = EXAMPLE_DIR / "Input" / "sacva_fd_eqcom"
CG_OUTPUT = EXAMPLE_DIR / "Output" / "sacva_cg_aad_eqcom"
FD_AAD_OUTPUT = EXAMPLE_DIR / "Output" / "sacva_fd_aad_eqcom"
FD_BUMP_OUTPUT = EXAMPLE_DIR / "Output" / "sacva_fd_eqcom"

print("CG master:", CG_MASTER)
print("FD AAD master:", FD_AAD_MASTER)
print("FD bump master:", FD_BUMP_MASTER)
print("CG input:", CG_INPUT)
print("FD input:", FD_INPUT)
print("CG output:", CG_OUTPUT)
print("FD AAD output:", FD_AAD_OUTPUT)
print("FD bump output:", FD_BUMP_OUTPUT)


CG master: /Users/fordesmith/Documents/examples/Notebooks/Example_11/Input/vre_sacva_cg_ad_eqcom.xml
FD AAD master: /Users/fordesmith/Documents/examples/Notebooks/Example_11/Input/vre_sacva_fd_aad_eqcom.xml
FD bump master: /Users/fordesmith/Documents/examples/Notebooks/Example_11/Input/vre_sacva_fd_eqcom.xml
CG input: /Users/fordesmith/Documents/examples/Notebooks/Example_11/Input/sacva_cg_eqcom
FD input: /Users/fordesmith/Documents/examples/Notebooks/Example_11/Input/sacva_fd_eqcom
CG output: /Users/fordesmith/Documents/examples/Notebooks/Example_11/Output/sacva_cg_aad_eqcom
FD AAD output: /Users/fordesmith/Documents/examples/Notebooks/Example_11/Output/sacva_fd_aad_eqcom
FD bump output: /Users/fordesmith/Documents/examples/Notebooks/Example_11/Output/sacva_fd_eqcom


## How the SA-CVA CG pipeline flows

```
[1] Config load
    master XML -> pricingengine_amccg.xml -> todaysmarket.xml -> curveconfig.xml
    simulation.xml + sensimarket.xml + xvasensiconfig.xml + sensitivity.xml
      |
[2] Market build
    quotes/fixings -> Market + ScenarioGenerator (IR/FX/EQ/COM/default grids)
      |
[3] Factor mapping
    maps trade risk keys to simulated factors (spot/vol/curve IDs, tenors, buckets)
      |
[4] CG build
    ScriptedTrade graph -> GaussianCamCG (measure/midpoint/compact/lambda)
      |
[5] Forward pass
    MC paths -> NPVs/exposure per scenario/date; CAM writes exposure cube
      |
[6] Reverse pass (AAD)
    adjoints on the same paths -> pathwise Greeks -> aggregated sensi cube
      |
[7] SA-CVA aggregation
    cva_sensitivities.csv + sacva_sensitivities.csv (credit/IR/EQ/COM/FX)
    sacva.csv (capital), xva_exposure.csv (EPE/ENE), cg_trace*.csv (debug)
```

**SaaS / downstream**
```
Outputs -> upload
  sacva.csv, sacva_sensitivities.csv, cva_sensitivities.csv, xva_exposure.csv
    |
  BigQuery landing tables
    |
  Post-processing: API surfaces, raw SQL, AI Q&A, trend/risk alerts
```

Alignment tip: every arrow depends on consistent IDs/tenors; if grids drift, the CG sensi cube and SA-CVA aggregation diverge.

## Run both examples from here

Use the cells below to execute the CG/AAD and FD runs directly from the notebook (uses the current Python kernel and venv). Set `USE_GPU = True` to patch the XMLs for your GPU via `EXTERNAL_COMPUTE_DEVICE` if desired.

In [3]:
import os
import sys
import time
import subprocess
from pathlib import Path
import ipywidgets as widgets
from IPython.display import display

try:
    NOTEBOOK_DIR = Path(__file__).resolve().parent
except NameError:
    NOTEBOOK_DIR = Path.cwd()

USE_GPU = False
BUMP_SCRIPT = NOTEBOOK_DIR / "run_sacva_eqcom_fd.py"
BUMP_OUT = NOTEBOOK_DIR / "Output" / "sacva_fd_eqcom"


def _latest_progress(log_path: Path) -> int | None:
    if not log_path.exists():
        return None
    max_pct = None
    try:
        with open(log_path, "r") as f:
            for line in f:
                if "XVA: Building cube" in line and "%" in line:
                    try:
                        pct = int(line.split("%")[0].split("(")[-1])
                        max_pct = pct if max_pct is None else max(max_pct, pct)
                    except Exception:
                        pass
    except Exception:
        return max_pct
    return max_pct


def run_fd_bump_with_progress():
    progress = widgets.IntProgress(value=0, min=0, max=100, description='FD bump', bar_style='info')
    display(progress)
    cmd = [sys.executable, str(BUMP_SCRIPT)]
    env = os.environ.copy()
    env.setdefault("PYTHONUNBUFFERED", "1")
    if USE_GPU:
        env["USE_EXTERNAL_COMPUTE_DEVICE"] = env.get("EXTERNAL_COMPUTE_DEVICE", "")
    print(">>> {}".format(" ".join(cmd)))
    proc = subprocess.Popen(cmd, cwd=NOTEBOOK_DIR, env=env, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    log_path = BUMP_OUT / "log_fd_eqcom.txt"
    try:
        while True:
            if proc.stdout:
                line = proc.stdout.readline()
                if line:
                    print(line, end="")
            code = proc.poll()
            pct = _latest_progress(log_path)
            if pct is not None:
                progress.value = min(100, pct)
            if code is not None:
                break
            time.sleep(0.5)
    finally:
        try:
            proc.wait(timeout=5)
        except Exception:
            proc.kill()
    progress.bar_style = 'success' if proc.returncode == 0 else 'danger'
    progress.value = 100

run_fd_bump_with_progress()


IntProgress(value=0, bar_style='info', description='FD bump')

>>> /usr/local/bin/python3 /Users/fordesmith/Documents/examples/Notebooks/Example_11/run_sacva_eqcom_fd.py


+-----------------------------------------------------+


| XVA Risk: SA-CVA    - FD bump 2 Trade EQ COM       |


+-----------------------------------------------------+


[fd-bump] master xml: /Users/fordesmith/Documents/examples/Notebooks/Example_11/Input/vre_sacva_fd_eqcom.xml


[fd-bump] outputs: /Users/fordesmith/Documents/examples/Notebooks/Example_11/Output


- cashflow


In [4]:
import os
import sys
import time
import subprocess
from pathlib import Path
import ipywidgets as widgets
from IPython.display import display

try:
    NOTEBOOK_DIR = Path(__file__).resolve().parent
except NameError:
    NOTEBOOK_DIR = Path.cwd()

USE_GPU = False
AAD_SCRIPT = NOTEBOOK_DIR / "run_sacva_eqcom_fd_aad.py"
AAD_OUT = NOTEBOOK_DIR / "Output" / "sacva_fd_aad_eqcom"


def _latest_progress(log_path: Path) -> int | None:
    if not log_path.exists():
        return None
    max_pct = None
    try:
        with open(log_path, "r") as f:
            for line in f:
                if "XVA: Building cube" in line and "%" in line:
                    try:
                        pct = int(line.split("%")[0].split("(")[-1])
                        max_pct = pct if max_pct is None else max(max_pct, pct)
                    except Exception:
                        pass
    except Exception:
        return max_pct
    return max_pct


def run_fd_aad_with_progress():
    progress = widgets.IntProgress(value=0, min=0, max=100, description='FD+AAD', bar_style='info')
    display(progress)
    cmd = [sys.executable, str(AAD_SCRIPT)]
    env = os.environ.copy()
    env.setdefault("PYTHONUNBUFFERED", "1")
    if USE_GPU:
        env["USE_EXTERNAL_COMPUTE_DEVICE"] = env.get("EXTERNAL_COMPUTE_DEVICE", "")
    print(">>> {}".format(" ".join(cmd)))
    proc = subprocess.Popen(cmd, cwd=NOTEBOOK_DIR, env=env, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    log_path = AAD_OUT / "log_fd_aad_eqcom.txt"
    try:
        while True:
            if proc.stdout:
                line = proc.stdout.readline()
                if line:
                    print(line, end="")
            code = proc.poll()
            pct = _latest_progress(log_path)
            if pct is not None:
                progress.value = min(100, pct)
            if code is not None:
                break
            time.sleep(0.5)
    finally:
        try:
            proc.wait(timeout=5)
        except Exception:
            proc.kill()
    progress.bar_style = 'success' if proc.returncode == 0 else 'danger'
    progress.value = 100

run_fd_aad_with_progress()


IntProgress(value=0, bar_style='info', description='FD+AAD')

>>> /usr/local/bin/python3 /Users/fordesmith/Documents/examples/Notebooks/Example_11/run_sacva_eqcom_fd_aad.py


+-----------------------------------------------------+


| XVA Risk: SA-CVA    - AAD (CG) 2 Trade EQ COM      |


+-----------------------------------------------------+


[fd-aad] master xml: /Users/fordesmith/Documents/examples/Notebooks/Example_11/Input/vre_sacva_fd_aad_eqcom.xml


## Inspect CG outputs

List the key files produced by the CG run and preview the first rows inline.

In [5]:
from pathlib import Path
import pandas as pd

cg_dir = CG_OUTPUT  # set earlier
print("CG output dir:", cg_dir)
files = [
    ("sacva.csv", "SA-CVA capital summary"),
    ("sacva_sensitivities.csv", "SA-CVA sensitivities"),
    ("cva_sensitivities.csv", "CAM CVA sensitivities"),
    ("xva_exposure.csv", "Exposure profile (EPE/ENE)")
]
available = []
for name, desc in files:
    p = Path(cg_dir) / name
    status = "FOUND" if p.exists() else "missing"
    print(f"- {name:25s} {status:7s} : {desc}")
    if p.exists():
        available.append(p)
for extra in sorted(Path(cg_dir).glob("cg_trace*.csv")):
    print(f"- {extra.name:25s} FOUND    : Sensi trace (debug)")
    available.append(extra)

TARGET = available[0] if available else None  # pick first available; change manually if desired
if TARGET:
    print("\nPreviewing:", TARGET.name)
    if TARGET.suffix.lower() == ".csv":
        display(pd.read_csv(TARGET).head(20))
    else:
        print(TARGET.read_text()[:2000])
else:
    print("\nRun the CG job to populate outputs.")

CG output dir: /Users/fordesmith/Documents/examples/Notebooks/Example_11/Output/sacva_cg_aad_eqcom
- sacva.csv                 missing : SA-CVA capital summary
- sacva_sensitivities.csv   missing : SA-CVA sensitivities
- cva_sensitivities.csv     missing : CAM CVA sensitivities
- xva_exposure.csv          missing : Exposure profile (EPE/ENE)

Run the CG job to populate outputs.


## Quick diff on outputs (run after the jobs finish)

In [6]:
import pandas as pd
from pathlib import Path
from pandas.api.types import is_numeric_dtype

# Files to compare
file_pairs = [
    ("sacva.csv", "sacva.csv"),
    ("sacva_sensitivities.csv", "sacva_sensitivities.csv"),
    ("xvacg-exposure.csv", "xvacg-exposure.csv"),
    ("xva_exposure.csv", "xva_exposure.csv"),
    ("cva_sensitivities.csv", "cva_sensitivities.csv"),
]

names_seen: set[str] = set()

def _to_numeric(series: pd.Series) -> pd.Series:
    num = pd.to_numeric(series, errors="coerce")
    if not is_numeric_dtype(num):
        num = num.astype(float)
    return num

def _safe_delta(lhs: pd.Series, rhs: pd.Series) -> pd.Series:
    try:
        return (lhs - rhs).abs().dropna()
    except Exception:
        lhs_num = _to_numeric(lhs)
        rhs_num = _to_numeric(rhs)
        try:
            return (lhs_num - rhs_num).abs().dropna()
        except Exception:
            return pd.Series(dtype=float)

def _collect_stats(cg: pd.DataFrame, fd: pd.DataFrame, keys: list[str]):
    stats = []
    if keys:
        merged = cg.merge(fd, on=keys, suffixes=("_cg", "_fd"))
        for col in merged.columns:
            if not col.endswith("_cg"):
                continue
            peer = col[:-3] + "_fd"
            if peer not in merged:
                continue
            lhs = _to_numeric(merged[col])
            rhs = _to_numeric(merged[peer])
            delta = _safe_delta(lhs, rhs)
            if not delta.empty:
                stats.append((col[:-3], float(delta.max()), float(delta.mean())))
    else:
        common = [c for c in cg.columns if c in fd.columns]
        for col in common:
            lhs = _to_numeric(cg[col])
            rhs = _to_numeric(fd[col])
            delta = _safe_delta(lhs, rhs)
            if not delta.empty:
                stats.append((col, float(delta.max()), float(delta.mean())))
    return stats

for cg_name, fd_name in file_pairs:
    if cg_name in names_seen:
        continue
    names_seen.add(cg_name)
    cg_path = FD_BUMP_OUTPUT / cg_name
    fd_path = FD_AAD_OUTPUT / fd_name
    print(cg_name)
    if not cg_path.exists() or not fd_path.exists():
        print(f"  run the jobs to create both outputs ({cg_path}, {fd_path})")
        continue
    try:
        cg = pd.read_csv(cg_path)
        fd = pd.read_csv(fd_path)
    except Exception as e:
        print("  could not read CSV:", e)
        continue
    if set(cg.columns) != set(fd.columns):
        print("  column mismatch; showing bump columns only")
        print(sorted(cg.columns))
        continue
    keys = [k for k in ["TradeId", "Type", "Bucket", "Factor"] if k in cg.columns]
    stats = _collect_stats(cg, fd, keys)
    if stats:
        stats.sort(key=lambda x: x[1], reverse=True)
        print("  max/mean abs diff per column:")
        for name_, mx, mean in stats:
            print(f"    {name_:20s} max={mx:.4g} mean={mean:.4g}")
    else:
        print("  no numeric columns to compare")


sacva.csv
  max/mean abs diff per column:
    Value                max=2.583e+05 mean=1.023e+05
sacva_sensitivities.csv
  max/mean abs diff per column:
    Value                max=4.3e+06 mean=1.109e+06
xvacg-exposure.csv
  run the jobs to create both outputs (/Users/fordesmith/Documents/examples/Notebooks/Example_11/Output/sacva_fd_eqcom/xvacg-exposure.csv, /Users/fordesmith/Documents/examples/Notebooks/Example_11/Output/sacva_fd_aad_eqcom/xvacg-exposure.csv)
xva_exposure.csv
  run the jobs to create both outputs (/Users/fordesmith/Documents/examples/Notebooks/Example_11/Output/sacva_fd_eqcom/xva_exposure.csv, /Users/fordesmith/Documents/examples/Notebooks/Example_11/Output/sacva_fd_aad_eqcom/xva_exposure.csv)
cva_sensitivities.csv
  max/mean abs diff per column:
    BaseCva              max=1.396e+04 mean=1.396e+04
    Delta                max=5021 mean=770.7
    ShiftSize            max=21.47 mean=2.433


## Reading residual differences
- Credit and IR: both grids use credit tenors 6M/1Y/5Y and IR tenors 1Y/5Y; small gaps usually come from CAM recalibration vs FD bumping. Increase samples/time steps if variance dominates.
- EQ/COM vegas: CG relies on vol surfaces in `sensimarket.xml`; if you toggle sim vols off, vegas will drop to zero. Keep EQ/COM vols on for parity runs.
- FX orientation and CSA: this example is uncollateralised with EUR base currency and EURUSD orientation; changing CSA or base currency will move CVA levels.
- Measure/scheme: domestic + midpoint + compact solve are the default parity set; try the lambda sweep or drift gate only when debugging model differences.